In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("MyApp").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("airline_2m.csv", header=True, inferSchema=True)

# Xem cấu trúc bảng
df.printSchema()
#Xem các tên cột
print(df.columns)

#Xem các năm có trong dữ liệu
df.select("Year").distinct().sort("Year", ascending=False).show()

# 1.1 Tách file dữ liệu thành file dữ liệu cho 12 tháng trong năm 2020,
# sau đó đọc các file dữ liệu chuyến bay theo tháng vào spark,
# hiển thị schema và tổng số chuyến bay

df.filter(F.col("Year") == 2020) \
  .write.partitionBy("Month") \
  .mode("overwrite") \
  .parquet("data_2020_by_month")

df_2020 = spark.read.parquet("data_2020_by_month")

#Kết quả
df_2020.printSchema()
print(f"Tổng số chuyến bay năm 2020: {df_2020.count()}")

# Đếm số chuyến bay theo từng tháng của năm 2020
df.filter(F.col("Year") == 2020).groupBy("Month").count().orderBy("Month").show()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 04:23:42 WARN Utils: Your hostname, nxbach, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 04:23:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 04:23:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true)
 |-- DestAirportSeqID: 

+----+
|Year|
+----+
|2020|
|2019|
|2018|
|2017|
|2016|
|2015|
|2014|
|2013|
|2012|
|2011|
|2010|
|2009|
|2008|
|2007|
|2006|
|2005|
|2004|
|2003|
|2002|
|2001|
+----+
only showing top 20 rows


root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true)
 |-- DestAirportSeqID: integer (nullable = true)
 |-- DestCit

+-----+-----+
|Month|count|
+-----+-----+
|    1| 6332|
|    2| 5936|
|    3| 6637|
+-----+-----+



In [2]:
#1.2 Tính số chuyến bay theo hãng (Reporting_Airline hoặc cột tương đương)

df.groupBy("Reporting_Airline").count().orderBy("count", ascending=False).show()


+-----------------+------+
|Reporting_Airline| count|
+-----------------+------+
|               WN|306238|
|               DL|264455|
|               AA|234730|
|               UA|194294|
|               US|172532|
|               NW|109336|
|               OO|107153|
|               CO| 90931|
|               MQ| 78113|
|               EV| 67600|
|               AS| 49656|
|               TW| 38531|
|               B6| 37638|
|               HP| 37630|
|               XE| 35645|
|               FL| 25954|
|               OH| 24786|
|               YV| 22555|
|               9E| 19666|
|               F9| 14320|
+-----------------+------+
only showing top 20 rows


In [3]:
#1.3 Tính tỉ lệ chuyến bay đúng giờ, trễ, hủy, chuyển hướng
total = df.count()

df.select(
    (F.sum(F.when(F.col("ArrDelay") <= 0, 1).otherwise(0)) / total * 100).alias("DungGio_%"),
    (F.sum(F.when(F.col("ArrDelay") > 0, 1).otherwise(0)) / total * 100).alias("Tre_%"),
    (F.sum(F.when(F.col("Cancelled") == 1, 1).otherwise(0)) / total * 100).alias("Huy_%"),
    (F.sum(F.when(F.col("Diverted") == 1, 1).otherwise(0)) / total * 100).alias("ChuyenHuong_%")
).show()



+------------------+-------+------------------+-------------+
|         DungGio_%|  Tre_%|             Huy_%|ChuyenHuong_%|
+------------------+-------+------------------+-------------+
|54.968399999999995|42.9777|1.8231000000000002|       0.2295|
+------------------+-------+------------------+-------------+



In [4]:
#1.4 Tìm 20 sân bay xuất phát có số chuyến nhiều nhất
df.groupBy("Origin").count().orderBy(F.desc("count")).show(20)


+------+------+
|Origin| count|
+------+------+
|   ATL|108667|
|   ORD|103263|
|   DFW| 88774|
|   LAX| 66769|
|   DEN| 60946|
|   PHX| 55573|
|   IAH| 49379|
|   SFO| 46444|
|   DTW| 46134|
|   LAS| 44797|
|   MSP| 43649|
|   CLT| 43195|
|   EWR| 41794|
|   BOS| 37510|
|   LGA| 37003|
|   MCO| 34505|
|   STL| 34324|
|   SLC| 34007|
|   SEA| 33810|
|   PHL| 32510|
+------+------+
only showing top 20 rows


In [5]:
#1.5 Tìm 20 sân bay có số chuyến nhiều nhất
df.groupBy("Dest").count().orderBy(F.desc("count")).show(20)

# Xuất phát
df_origin = df.select(F.col("Origin").alias("Airport"))
# Đến
df_dest = df.select(F.col("Dest").alias("Airport"))

# Gộp xuất phát và đến thành tổng
total = df_origin.union(df_dest)
total.groupBy("Airport").count().sort("count", ascending=False).show(20)


+----+------+
|Dest| count|
+----+------+
| ATL|108112|
| ORD|103203|
| DFW| 88620|
| LAX| 66722|
| DEN| 60998|
| PHX| 56200|
| IAH| 49345|
| SFO| 46883|
| DTW| 46382|
| LAS| 44557|
| MSP| 43404|
| CLT| 43099|
| EWR| 41376|
| BOS| 37639|
| LGA| 36995|
| MCO| 34857|
| STL| 34841|
| SLC| 33964|
| SEA| 33944|
| PHL| 32091|
+----+------+
only showing top 20 rows


+-------+------+
|Airport| count|
+-------+------+
|    ATL|216779|
|    ORD|206466|
|    DFW|177394|
|    LAX|133491|
|    DEN|121944|
|    PHX|111773|
|    IAH| 98724|
|    SFO| 93327|
|    DTW| 92516|
|    LAS| 89354|
|    MSP| 87053|
|    CLT| 86294|
|    EWR| 83170|
|    BOS| 75149|
|    LGA| 73998|
|    MCO| 69362|
|    STL| 69165|
|    SLC| 67971|
|    SEA| 67754|
|    PHL| 64601|
+-------+------+
only showing top 20 rows


In [6]:
#1.6 Tính thời gian trễ khởi hành trung bình và trễ đến trung bình theo hãng
df.groupBy("Reporting_Airline").agg(
    F.avg("DepDelay").alias("Avg_Tre_KhoiHanh"),
    F.avg("ArrDelay").alias("Avg_Tre_Den")
).orderBy("Avg_Tre_Den", ascending=False).show()


+-----------------+------------------+------------------+
|Reporting_Airline|  Avg_Tre_KhoiHanh|       Avg_Tre_Den|
+-----------------+------------------+------------------+
|               PI| 10.29743473838852|10.905771629436444|
|               PS| 9.660069848661234| 10.23076923076923|
|               G4|12.006191950464396|10.031527531083482|
|               EV|12.664960156023831| 9.315618934079202|
|               B6|12.999784058088375| 8.966919328641039|
|               XE|  8.59135402015099|  8.72189741078886|
|               YV|10.441027281451355| 8.410168248292303|
|               OH|10.727890589167327| 8.205267558528428|
|               F9|10.251869092960925| 7.936846566826787|
|               HP| 8.315420119301466| 7.736314238598578|
|               MQ|  8.97980013040759| 7.720386042662253|
|               UA|10.099442964144094| 7.163953304937687|
|               DH| 9.763824884792626| 7.136173285198556|
|               FL|  8.66549966935076|6.9477379095163805|
|             

In [7]:
# 1.7 Tìm 10 tuyến bay (Origin-Dest) có số chuyến bị trễ nhiều nhất
# Lọc những chuyến bị trễ (ArrDelay > 0) rồi đếm
df.filter(F.col("ArrDelay") > 0) \
       .groupBy("Origin", "Dest").count() \
       .orderBy(F.desc("count")) \
       .show(10)


+------+----+-----+
|Origin|Dest|count|
+------+----+-----+
|   SFO| LAX| 2410|
|   LAX| SFO| 2264|
|   LAS| LAX| 2202|
|   LAX| LAS| 2129|
|   PHX| LAX| 1871|
|   LAX| PHX| 1867|
|   PHX| LAS| 1609|
|   ORD| LGA| 1557|
|   LAS| PHX| 1473|
|   ORD| MSP| 1462|
+------+----+-----+
only showing top 10 rows


In [8]:
# 1.8 Tính tổng thời gian trễ theo từng nguyên nhân nếu có các cột nguyên nhân delay
# Danh sách các cột nguyên nhân phổ biến
delay_causes = ["WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

# Chỉ chọn những cột tồn tại trong dữ liệu
existing_causes = [c for c in delay_causes if c in df.columns]

if existing_causes:
    df.select([F.sum(c).alias(f"Total_{c}") for c in existing_causes]).show()
else:
    print("Dữ liệu không chứa các cột nguyên nhân chi tiết.")


+------------------+--------------+-------------------+-----------------------+
|Total_WeatherDelay|Total_NASDelay|Total_SecurityDelay|Total_LateAircraftDelay|
+------------------+--------------+-------------------+-----------------------+
|          652085.0|     3413414.0|            18825.0|              4891681.0|
+------------------+--------------+-------------------+-----------------------+



In [9]:
# 1.9 Tạo cột mới "delay_level": nếu ArrDelay <= 0 là on_time,
# từ 1 đến 30 là minor_delay,
# từ 31 đến 120 là major_delay,
# còn lại là severe_delay
df = df.withColumn("delay_level", 
    F.when(F.col("ArrDelay") <= 0, "on_time")
     .when((F.col("ArrDelay") >= 1) & (F.col("ArrDelay") <= 30), "minor_delay")
     .when((F.col("ArrDelay") >= 31) & (F.col("ArrDelay") <= 120), "major_delay")
     .otherwise("severe_delay")
)
# Kết quả
df.select("ArrDelay", "delay_level").show(40)


+--------+------------+
|ArrDelay| delay_level|
+--------+------------+
|    23.0| minor_delay|
|     0.0|     on_time|
|    -3.0|     on_time|
|   -20.0|     on_time|
|    32.0| major_delay|
|    11.0| minor_delay|
|     2.0| minor_delay|
|   214.0|severe_delay|
|    10.0| minor_delay|
|    29.0| minor_delay|
|     6.0| minor_delay|
|    -5.0|     on_time|
|   -10.0|     on_time|
|   -19.0|     on_time|
|   -16.0|     on_time|
|     9.0| minor_delay|
|    -7.0|     on_time|
|   -23.0|     on_time|
|   -14.0|     on_time|
|    NULL|severe_delay|
|   -12.0|     on_time|
|    -2.0|     on_time|
|     8.0| minor_delay|
|    44.0| major_delay|
|    17.0| minor_delay|
|     6.0| minor_delay|
|   -16.0|     on_time|
|   -25.0|     on_time|
|    -2.0|     on_time|
|    39.0| major_delay|
|    -3.0|     on_time|
|    -6.0|     on_time|
|    -5.0|     on_time|
|   -22.0|     on_time|
|    -6.0|     on_time|
|    -8.0|     on_time|
|    NULL|severe_delay|
|   -25.0|     on_time|
|   -19.0|     o

In [10]:
# 1.10 Tạo cột mới "flight_period" theo giờ khởi hành thực tế: sáng, chiều, tối, đêm'
# Lấy giờ bằng cách chia cho 100
df = df.withColumn("DepHour", (F.col("DepTime") / 100).cast("int"))

df = df.withColumn("flight_period", 
    F.when((F.col("DepHour") >= 5) & (F.col("DepHour") <= 11), "sáng")
     .when((F.col("DepHour") >= 12) & (F.col("DepHour") <= 17), "chiều")
     .when((F.col("DepHour") >= 18) & (F.col("DepHour") <= 22), "tối")
     .otherwise("đêm")
)
# Kết quả
df.select("DepTime", "DepHour", "flight_period").show(10)

+-------+-------+-------------+
|DepTime|DepHour|flight_period|
+-------+-------+-------------+
|   1659|     16|        chiều|
|   1202|     12|        chiều|
|   1644|     16|        chiều|
|   1305|     13|        chiều|
|   1911|     19|          tối|
|    639|      6|         sáng|
|   1751|     17|        chiều|
|   2331|     23|          đêm|
|   1552|     15|        chiều|
|   2046|     20|          tối|
+-------+-------+-------------+
only showing top 10 rows


In [11]:
#Câu 4: Sử dụng chuỗi file theo tháng của năm 2020
#để thực hành streaming top 5 hãng hàng không có số chuyến bay nhiều nhất theo ngày, 
# hiển thị airline, flight_date, flight_count, avg_arr_delay

# 1. Định nghĩa Schema
file_schema = df_2020.schema

# 2. Thiết lập Streaming Source
streaming_df = spark.readStream \
    .schema(file_schema) \
    .option("maxFilesPerTrigger", 1) \
    .parquet("data_2020_by_month")

# 3. Xử lý dữ liệu
# Tính toán Top 5 hãng theo ngày
result_stream = streaming_df.groupBy("FlightDate", "Reporting_Airline") \
    .agg(
        F.count("*").alias("flight_count"),
        F.avg("ArrDelay").alias("avg_arr_delay")
    )

# 4. Ghi kết quả ra Console (Có thêm checkpointLocation để tránh lỗi Py4J)
import shutil
import os

# Xóa thư mục checkpoint cũ nếu có để chạy mới hoàn toàn
checkpoint_dir = "checkpoint_test"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)

query = result_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime='5 seconds') \
    .start()

In [ ]:
#Dừng streaming
# query.stop()